# TAMPO-GCN — Colab Test Run Notebook

This notebook tests every component of the GDRL GCN integration and the benchmarking framework.  
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [ ]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/tampo.git tampo
%cd tampo

In [3]:
!nvidia-smi

Tue Jun  9 07:54:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [ ]:
# ── 0c. Verify imports ────────────────────────────────────────────
import torch, gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data
print("torch:", torch.__version__)
print("gym:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Unit Test — DAGEncoder with GCN path

Verifies that the encoder:
- Accepts a PyG Batch of **variable-size** graphs
- Produces a context vector of the correct shape 
- Requires no  call

In [ ]:
import sys, os
sys.path.insert(0, '.')  # tampo root

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM = 6
HIDDEN_DIM = 128
SERVER_FEATURE_DIM = 20

encoder = DAGEncoder(
    task_feature_dim=TASK_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=2,
    encoder_type='gcn',
    server_feature_dim=SERVER_FEATURE_DIM
)
encoder.eval()

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    # Random directed edges (roughly a chain)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)

# Dummy task_features tensor (content ignored in GCN path, only shape used)
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features = torch.rand(3, SERVER_FEATURE_DIM)

with torch.no_grad():
    encoded_tasks, context = encoder(
        task_features=task_features_dummy,
        graph_batch=batch,
        server_features=server_features
    )

assert context.shape == (3, HIDDEN_DIM * 2), f"Bad context shape: {context.shape}"
print(f"PASS — context shape: {context.shape}  (expected [3, {HIDDEN_DIM*2}])")
print(f"PASS — encoded_tasks shape: {encoded_tasks.shape}")


## 2. Unit Test — FlatVectorWrapper observation shape

In [ ]:
import yaml, sys
sys.path.insert(0, '.')

from env.base_offloading_env import TaskOffloadingEnv
from env.wrappers.flat_vector_wrapper import FlatVectorWrapper

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)

cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc['environment'][sec])

env = TaskOffloadingEnv(cfg)
wrapped = FlatVectorWrapper(env, max_tasks=30, task_feature_dim=6)

obs = wrapped.reset()
expected_dim = 30 * 6 + 20   # max_tasks * feat_dim + server_dim
assert obs.shape == (expected_dim,), f"Expected ({expected_dim},), got {obs.shape}"
print(f"PASS — flat obs shape: {obs.shape}")


## 3. Unit Test — SequenceWrapper topological order

In [ ]:
import numpy as np, sys
sys.path.insert(0, '.')

from env.base_offloading_env import TaskOffloadingEnv
from env.wrappers.sequence_wrapper import SequenceWrapper
import yaml

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)

cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc['environment'][sec])

env = TaskOffloadingEnv(cfg)
wrapped = SequenceWrapper(env, max_tasks=30, task_feature_dim=6)

obs = wrapped.reset()
assert obs.shape == (30, 6), f"Expected (30, 6), got {obs.shape}"
print(f"PASS — sequence obs shape: {obs.shape}")

# Verify topological order on a known DAG
from env.wrappers.sequence_wrapper import SequenceWrapper
adj = np.array([
    [0,1,1,0],
    [0,0,0,1],
    [0,0,0,1],
    [0,0,0,0],
], dtype=float)
order = SequenceWrapper._topological_sort(adj)
# Node 0 must come before 1 and 2; nodes 1,2 before 3
assert order.index(0) < order.index(1)
assert order.index(0) < order.index(2)
assert order.index(1) < order.index(3)
assert order.index(2) < order.index(3)
print(f"PASS — topological order: {order}")


## 4. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [ ]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 5. Quick Training Smoke Test — TAMPO-GCN (1 iteration)

Checks that the full training loop executes without shape errors.

In [ ]:
import yaml, sys
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv

cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(full_config['environment'][sec])

env = TaskOffloadingEnv(cfg)

# Load the generated dataset
import json
with open('data/test_dags.json') as f:
    test_dags = json.load(f)
env.load_task_dataset(test_dags)

tampo_cfg = full_config['algorithms']['tampo'].copy()
tampo_cfg['encoder_type'] = 'gcn'
tampo_cfg['num_meta_iterations'] = 1  # just 1 iteration for smoke test

from algorithms.rl.tampo import TAMPOFramework
framework = TAMPOFramework(env, tampo_cfg)

print('
Running 1 meta-iteration smoke test...')
results = framework.train(num_meta_iterations=1, num_tasks_per_iter=2)
print('
SMOKE TEST PASSED — no shape errors.')
print('Results keys:', list(results.keys()) if isinstance(results, dict) else type(results))


## 6. Run Benchmark on Trained Models

After training (in Step 5 or via ), save checkpoint then benchmark.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Save checkpoint from smoke test (in real runs, main.py saves this automatically)
os.makedirs('models', exist_ok=True)
# framework.save('models/tampo_gcn_checkpoint.pth')  # uncomment if framework is trained

# Run benchmark
!python benchmark.py \
    --algorithms TAMPO_GCN \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/


## 7. Download Results

In [ ]:
from google.colab import files
import os

for fname in ['results/comparison_bar.png', 'results/pareto_front.png', 'results/benchmark_results.csv']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded {fname}')
    else:
        print(f'Not found (run benchmark first): {fname}')


## Checklist

- [ ] Step 1 PASS — DAGEncoder context shape 
- [ ] Step 2 PASS — FlatVectorWrapper shape 
- [ ] Step 3 PASS — SequenceWrapper shape  + topo order verified
- [ ] Step 4 — Dataset JSON created
- [ ] Step 5 PASS — Training smoke test completes without error
- [ ] Step 6 — Benchmark CSV and plots generated
- [ ] Step 7 — Files downloaded